In [ ]:
import pandas as pd
import glob
from utils import *

In [ ]:
dfs = []
for file in glob.glob(f"../results/experiment*solvers*.csv"):
    dfs.append(pd.read_csv(file))
df = pd.concat(dfs, ignore_index=True)
df = process_experiments_df(df)
df

,datetime,test case,p,mesh family,fine m,solvers m,coarse m,solver,random rhs,solution warmup steps,...,exception,residual norm,metadata,setup metadata,solve times,matrix and DOF map copy times,matrix permute times,solver setup times,rhs copy and permute times,solution permute and copy times
0,2025-12-15T18:15:41.629731+00:00,continuous coefficient 3D,1,"uniform(3D,7)",S7,-,-,"CG(AMGX(L1_TRUNC, torch.float32), bsr_matmul=T...",False,0,...,Conjugate gradient did not converge in 1200 it...,NaN,None,NaN,None,None,None,None,None,None
1,2025-12-15T18:16:13.083022+00:00,continuous coefficient 3D,1,"uniform(3D,7)",S7,-,-,"CG(AMGX(AGGRESIVE_L1, torch.float32), bsr_matm...",False,0,...,NaN,3.612740e-11,"{'iterations': 70, 'residual norms': [0.048546...",NaN,"[2407.5869140625, 2308.075927734375, 2308.0083...",[2538.906982421875],[0.014175999909639359],[2955.322021484375],[118.18077087402344],[112.63849639892578]
2,2025-12-15T18:16:44.474525+00:00,continuous coefficient 3D,1,"uniform(3D,7)",S7,-,-,"CG(AMGX(AGGRESSIVE_L1_TRUNC, torch.float32), b...",False,0,...,NaN,4.175830e-11,"{'iterations': 52, 'residual norms': [0.048546...",NaN,"[2496.060546875, 2398.398193359375, 2398.43041...",[1593.327880859375],[0.013472000136971474],[2984.781005859375],[38.41401672363281],[116.69750213623047]
3,2025-12-15T18:17:17.277707+00:00,continuous coefficient 3D,1,"uniform(3D,7)",S7,-,-,"CG(AMGX(AGGRESSIVE_CHEB_L1_TRUNC, torch.float3...",False,0,...,NaN,4.800101e-11,"{'iterations': 35, 'residual norms': [0.048546...",NaN,"[2645.398193359375, 2543.87255859375, 2545.697...",[1691.6640625],[0.01724799908697605],[2901.42431640625],[41.68668746948242],[114.72281646728516]
4,2025-12-16T11:00:28.608328+00:00,continuous coefficient 2D,5,unstructured(128),F,-,-,"CG(AMGX(L1_TRUNC, torch.float32), bsr_matmul=T...",False,0,...,NaN,2.105586e-10,"{'iterations': 94, 'residual norms': [0.214415...",NaN,"[3291.307373046875, 796.601806640625, 798.6620...",[202.8533477783203],[0.01679999940097332],[466.8438720703125],[9.188287734985352],[10.809887886047363]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
311,2025-12-16T10:44:41.560153+00:00,continuous coefficient 2D,1,unstructured(1000),F,C,C,"CG(HybridSchwarz(torch.float64, Inv(torch.floa...",False,0,...,NaN,4.210816e-11,"{'iterations': 38, 'residual norms': [9.427459...",NaN,"[872.9376220703125, 686.5474853515625, 686.396...",[496.3862609863281],[1517.9376220703125],[6836.85009765625],[17.652000427246094],[13.038816452026367]
312,2025-12-16T10:45:04.875874+00:00,continuous coefficient 2D,1,unstructured(1000),F,F,C,"CG(AdditiveSchwarz(torch.float32, Inv(torch.fl...",False,0,...,NaN,4.912479e-11,"{'iterations': 63, 'residual norms': [0.051607...",NaN,"[4232.87646484375, 1186.4990234375, 1180.35424...",[560.6296997070312],[1636.930419921875],[5540.18505859375],[15.447775840759277],[12.746623992919922]
313,2025-12-16T10:45:22.740495+00:00,continuous coefficient 2D,1,unstructured(1000),F,F,C,"CG(HybridSchwarz(torch.float64, Inv(torch.floa...",False,0,...,NaN,3.815321e-11,"{'iterations': 49, 'residual norms': [9.427459...",NaN,"[1234.1116943359375, 866.32470703125, 865.6563...",[408.9510803222656],[1591.6734619140625],[6134.51171875],[15.928671836853027],[12.492480278015137]
314,2025-12-16T10:46:16.874183+00:00,continuous coefficient 2D,1,unstructured(1000),F,F,F,"CG(AdditiveSchwarz(torch.float32, Inv(torch.fl...",False,0,...,NaN,4.054668e-11,"{'iterations': 58, 'residual norms': [0.051607...",NaN,"[4930.44921875, 3378.64990234375, 3380.1679687...",[552.09521484375],[1641.31787109375],[15821.4921875],[14.988832473754883],[12.566656112670898]


In [30]:
problems = (
    df[["test case", "p", "mesh family", "fine m"]]
    .drop_duplicates()
    .sort_values(by=["test case", "p", "mesh family", "fine m"])
    .reset_index(drop=True)
)

problems = problems[~problems["mesh family"].str.contains("unstructured")]

problems

,test case,p,mesh family,fine m
0,continuous coefficient 2D,1,"uniform(2D,11)",S11
1,continuous coefficient 2D,1,"uniform(2D,12)",S12
3,continuous coefficient 2D,3,"uniform(2D,10)",S10
5,continuous coefficient 2D,5,"uniform(2D,9)",S9
7,continuous coefficient 3D,1,"uniform(3D,7)",S7
8,continuous coefficient 3D,2,"uniform(3D,6)",S6
9,continuous coefficient 3D,3,"uniform(3D,5)",S5


In [31]:
def process_experiments_df(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["solve time"] = df["solve times"].apply(
        lambda times: (
            min(times) if times is not None and len(times) > 0 else float("inf")
        )
    )

    df = df[df.solver.str.contains("CG")].copy()

    summary = df.pivot_table(
        index=["solver"],
        columns=["coarse m", "solvers m"],
        values="solve time",
        aggfunc="min",
    )
    summary["best"] = summary.min(axis=1)

    best_amgx_key = summary["best"][summary.index.str.contains("AMGX")].idxmin()
    best_amgx_variant = best_amgx_key[8:].split(",")[0]
    best_amgx_time = summary["best"][best_amgx_key]

    best_additive_key = summary["best"][
        summary.index.str.contains("AdditiveSchwarz")
    ].idxmin()
    best_additive_time = summary["best"][best_additive_key]
    best_additive_by_meshes = summary[
        summary.index.str.contains("AdditiveSchwarz")
    ].min()
    best_additive_meshing = best_additive_by_meshes.idxmin()

    best_hybrid_key = summary["best"][
        summary.index.str.contains("HybridSchwarz")
    ].idxmin()
    best_hybrid_time = summary["best"][best_hybrid_key]
    best_hybrid_by_meshes = summary[summary.index.str.contains("HybridSchwarz")].min()
    best_hybrid_meshing = best_hybrid_by_meshes.idxmin()

    # parallel_mask = [b == (fine_m, "S") for a, b in best_by_meshes.index]
    # best_parallel_meshing = best_by_meshes[parallel_mask].idxmin()
    # best_parallel_time = best_by_meshes[best_parallel_meshing]
    # best_parallel_precision = best_by_meshes_prec[best_parallel_meshing]

    # classical_mask = [a == b for a, b in best_by_meshes.index]
    # best_classical_meshing = best_by_meshes[classical_mask].idxmin()
    # best_classical_time = best_by_meshes[best_classical_meshing]
    # best_classical_precision = best_by_meshes_prec[best_classical_meshing]

    return pd.Series(
        {
            "best_amgx_variant": best_amgx_variant,
            "best_amgx_time": best_amgx_time,
            "best_additive_time": best_additive_time,
            "best_additive_meshing": best_additive_meshing,
            "best_hybrid_time": best_hybrid_time,
            "best_hybrid_meshing": best_hybrid_meshing,
            # "best_asm_time": best_asm_time,
            # "best_asm_precision": best_asm_precision,
            # "best_parallel_meshing": best_parallel_meshing,
            # "best_parallel_time": best_parallel_time,
            # "best_parallel_precision": best_parallel_precision,
            # "best_classical_meshing": best_classical_meshing,
            # "best_classical_time": best_classical_time,
            # "best_classical_precision": best_classical_precision,
        }
    )

In [32]:
def get_dofs(d, fine_m, p):
    element_dofs = {
        # (d, p)
        (2, 1): 3,
        (2, 3): 10,
        (2, 5): 21,
        (3, 1): 4,
        (3, 2): 10,
        (3, 3): 20,
    }
    return 2 ** (d * fine_m) * (2 if d == 2 else 6) * element_dofs[(d, p)]

In [33]:
def get_problem_summary(problem):
    test_case = problem["test case"]
    p = problem["p"]
    mesh_family = problem["mesh family"]
    fine_m = problem["fine m"]

    data = df[
        (df["test case"] == test_case) & (df["p"] == p) & (df["fine m"] == fine_m) & (df["mesh family"] == mesh_family)
    ]
    if data.empty:
        return pd.Series()

    summary = process_experiments_df(data)
    d = int(test_case[-2])
    summary["d"] = d
    summary["mesh family"] = mesh_family
    summary["m"] = fine_m
    summary["p"] = p
    summary["DoFs"] = get_dofs(d, int(fine_m[1:]), p) if fine_m != "F" else None

    return summary

In [34]:
speedup_table = pd.DataFrame(
    get_problem_summary(problem) for _, problem in problems.iterrows()
)
speedup_table

,best_amgx_variant,best_amgx_time,best_additive_time,best_additive_meshing,best_hybrid_time,best_hybrid_meshing,d,mesh family,m,p,DoFs
0,L1_TRUNC,648.288696,665.173157,"(C9, S10)",425.881683,"(C10, C10)",2,"uniform(2D,11)",S11,1,25165824
1,L1_TRUNC,2567.355469,2476.706299,"(C10, S11)",1814.868408,"(C11, C12)",2,"uniform(2D,12)",S12,1,100663296
2,L1_TRUNC,1804.431885,1741.733398,"(S9, S9)",1039.380493,"(C10, C10)",2,"uniform(2D,10)",S10,3,20971520
3,L1_TRUNC,1794.323975,1782.699219,"(S9, S9)",1135.996704,"(C9, C9)",2,"uniform(2D,9)",S9,5,11010048
4,AGGRESIVE_L1,2307.943359,1799.299927,"(C6, C7)",1366.426758,"(C6, C7)",3,"uniform(3D,7)",S7,1,50331648
5,AGGRESSIVE_CHEB_L1_TRUNC,2229.325195,1510.241089,"(C5, C6)",1129.608887,"(C5, C6)",3,"uniform(3D,6)",S6,2,15728640
6,L1_TRUNC,1527.265991,820.082153,"(C5, C5)",567.428528,"(C5, C5)",3,"uniform(3D,5)",S5,3,3932160


In [35]:
# assert (speedup_table["best_asm_precision"] == "fp32+fp16").all()
# assert (speedup_table["best_parallel_precision"] == "fp32+fp16").all()
# assert (speedup_table["best_classical_precision"] == "fp32+fp16").all()

In [36]:
tab = pd.DataFrame()
tab[["$d$", "$p$"]] = speedup_table[["d", "p"]]
tab["$\\mathcal{T}_h$"] = speedup_table["m"].apply(lambda m: f"${format_mesh(m)}$")
tab["DoFs"] = speedup_table["DoFs"].apply(lambda n: f"{n / 1_000_000:.1f}\\,{{M}}")

amgx_best = "best AmgX configuration found"
additive_best = "best AdditiveSchwarz variant"
hybrid_best = "best HybridSchwarz variant"

tab[(amgx_best, "name")] = speedup_table["best_amgx_variant"].map(
    lambda v: f"\\verb|{v.replace('AGGRESIVE', 'AGGRESSIVE')}|"
)
tab[(amgx_best, "time")] = speedup_table["best_amgx_time"] / 1000
tab[(additive_best, "$\\mathcal{T}_\\mathcal{H}$")] = speedup_table[
    "best_additive_meshing"
].map(lambda m: f"${format_mesh(m[0])}$")
tab[(additive_best, "$\\mathcal{T}_H$")] = speedup_table["best_additive_meshing"].map(
    lambda m: f"${format_mesh(m[1])}$"
)
tab[(additive_best, "time")] = speedup_table["best_additive_time"] / 1000
tab[(additive_best, "speedup")] = (
    tab[(amgx_best, "time")] / tab[(additive_best, "time")]
)

tab[(hybrid_best, "$\\mathcal{T}_\\mathcal{H}$")] = speedup_table[
    "best_hybrid_meshing"
].map(lambda m: f"${format_mesh(m[0])}$")
tab[(hybrid_best, "$\\mathcal{T}_H$")] = speedup_table["best_hybrid_meshing"].map(
    lambda m: f"${format_mesh(m[1])}$"
)
tab[(hybrid_best, "time")] = speedup_table["best_hybrid_time"] / 1000
tab[(hybrid_best, "speedup")] = tab[(amgx_best, "time")] / tab[(hybrid_best, "time")]

tab.set_index(["$d$", "$p$", "$\\mathcal{T}_h$", "DoFs"], inplace=True)


def time_latex(t: float):
    return f"{t:.6f}\\, \\si{{\\second}}"


def speedup_latex(s: float):
    return f"{s:.6f} \\,\\(\\times\\)"


tab[(amgx_best, "time")] = tab[(amgx_best, "time")].apply(time_latex)
tab[(additive_best, "time")] = tab[(additive_best, "time")].apply(time_latex)
tab[(additive_best, "speedup")] = tab[(additive_best, "speedup")].apply(speedup_latex)
tab[(hybrid_best, "time")] = tab[(hybrid_best, "time")].apply(time_latex)
tab[(hybrid_best, "speedup")] = tab[(hybrid_best, "speedup")].apply(speedup_latex)

tab.columns = pd.MultiIndex.from_tuples(tab.columns)
tab.rename(columns=lambda c: f"{{{c}}}", inplace=True)
tab

{best AmgX configuration found}  \
                                                                {name}   
$d$ $p$ $\mathcal{T}_h$    DoFs                                          
2   1   $\mathcal{S}_{11}$ 25.2\,{M}                   \verb|L1_TRUNC|   
        $\mathcal{S}_{12}$ 100.7\,{M}                  \verb|L1_TRUNC|   
    3   $\mathcal{S}_{10}$ 21.0\,{M}                   \verb|L1_TRUNC|   
    5   $\mathcal{S}_{9}$  11.0\,{M}                   \verb|L1_TRUNC|   
3   1   $\mathcal{S}_{7}$  50.3\,{M}              \verb|AGGRESSIVE_L1|   
    2   $\mathcal{S}_{6}$  15.7\,{M}   \verb|AGGRESSIVE_CHEB_L1_TRUNC|   
    3   $\mathcal{S}_{5}$  3.9\,{M}                    \verb|L1_TRUNC|   

                                                                \
                                                        {time}   
$d$ $p$ $\mathcal{T}_h$    DoFs                                  
2   1   $\mathcal{S}_{11}$ 25.2\,{M}   0.648289\, \si{\second}   
        $\mathcal{S}_{12}$ 100.7\,{M}  2.567355\, \si{\second}   
    3   $\mathcal{S}_{10}$ 21.0\,{M}   1.804432\, \si{\second}   
    5   $\mathcal{S}_{9}$  11.0\,{M}   1.794324\, \si{\second}   
3   1   $\mathcal{S}_{7}$  50.3\,{M}   2.307943\, \si{\second}   
    2   $\mathcal{S}_{6}$  15.7\,{M}   2.229325\, \si{\second}   
    3   $\mathcal{S}_{5}$  3.9\,{M}    1.527266\, \si{\second}   

                                      {best AdditiveSchwarz variant}  \
                                         {$\mathcal{T}_\mathcal{H}$}   
$d$ $p$ $\mathcal{T}_h$    DoFs                                        
2   1   $\mathcal{S}_{11}$ 25.2\,{M}               $\mathcal{C}_{9}$   
        $\mathcal{S}_{12}$ 100.7\,{M}             $\mathcal{C}_{10}$   
    3   $\mathcal{S}_{10}$ 21.0\,{M}               $\mathcal{S}_{9}$   
    5   $\mathcal{S}_{9}$  11.0\,{M}               $\mathcal{S}_{9}$   
3   1   $\mathcal{S}_{7}$  50.3\,{M}               $\mathcal{C}_{6}$   
    2   $\mathcal{S}_{6}$  15.7\,{M}               $\mathcal{C}_{5}$   
    3   $\mathcal{S}_{5}$  3.9\,{M}                $\mathcal{C}_{5}$   

                                                           \
                                        {$\mathcal{T}_H$}   
$d$ $p$ $\mathcal{T}_h$    DoFs                             
2   1   $\mathcal{S}_{11}$ 25.2\,{M}   $\mathcal{S}_{10}$   
        $\mathcal{S}_{12}$ 100.7\,{M}  $\mathcal{S}_{11}$   
    3   $\mathcal{S}_{10}$ 21.0\,{M}    $\mathcal{S}_{9}$   
    5   $\mathcal{S}_{9}$  11.0\,{M}    $\mathcal{S}_{9}$   
3   1   $\mathcal{S}_{7}$  50.3\,{M}    $\mathcal{C}_{7}$   
    2   $\mathcal{S}_{6}$  15.7\,{M}    $\mathcal{C}_{6}$   
    3   $\mathcal{S}_{5}$  3.9\,{M}     $\mathcal{C}_{5}$   

                                                                \
                                                        {time}   
$d$ $p$ $\mathcal{T}_h$    DoFs                                  
2   1   $\mathcal{S}_{11}$ 25.2\,{M}   0.665173\, \si{\second}   
        $\mathcal{S}_{12}$ 100.7\,{M}  2.476706\, \si{\second}   
    3   $\mathcal{S}_{10}$ 21.0\,{M}   1.741733\, \si{\second}   
    5   $\mathcal{S}_{9}$  11.0\,{M}   1.782699\, \si{\second}   
3   1   $\mathcal{S}_{7}$  50.3\,{M}   1.799300\, \si{\second}   
    2   $\mathcal{S}_{6}$  15.7\,{M}   1.510241\, \si{\second}   
    3   $\mathcal{S}_{5}$  3.9\,{M}    0.820082\, \si{\second}   

                                                              \
                                                   {speedup}   
$d$ $p$ $\mathcal{T}_h$    DoFs                                
2   1   $\mathcal{S}_{11}$ 25.2\,{M}   0.974616 \,\(\times\)   
        $\mathcal{S}_{12}$ 100.7\,{M}  1.036601 \,\(\times\)   
    3   $\mathcal{S}_{10}$ 21.0\,{M}   1.035998 \,\(\times\)   
    5   $\mathcal{S}_{9}$  11.0\,{M}   1.006521 \,\(\times\)   
3   1   $\mathcal{S}_{7}$  50.3\,{M}   1.282690 \,\(\times\)   
    2   $\mathcal{S}_{6}$  15.7\,{M}   1.476139 \,\(\times\)   
    3   $\mathcal{S}_{5}$  3.9\,{M}    1.862333 \,\(\times\)   

   

In [37]:
time_col = "S[table-format=1.3\\,s, round-mode=places, round-precision=3]"
speedup_col = "S[table-format=1.3, round-mode=places, round-precision=3]"
dofs_col = "S[table-format=2.1\\,M]"
tab_latex = tab.style.to_latex(
    hrules=True,
    multirow_align="t",
    multicol_align="c",
    column_format=f"rrc{dofs_col}|l{time_col}|cc{time_col}{speedup_col}|cc{time_col}{speedup_col}",
)

In [38]:
custom_header = r"""
 &  &  &  & \multicolumn{2}{c|}{{best AmgX configuration found}} & \multicolumn{4}{c|}{{best additive variant considered}} & \multicolumn{4}{c}{{best hybrid variant considered}} \\
$d$ & $p$ & $\mathcal{T}_h$ & {DoFs} & {name} & {time} & {$\mathcal{T}_\mathcal{H}$} & {$\mathcal{T}_H$} & {time} & {speedup} & {$\mathcal{T}_\mathcal{H}$} & {$\mathcal{T}_H$} & {time} & {speedup} \\
"""

before_header = tab_latex.split("\\toprule")[0] + "\\toprule\n"
after_header = "\n\\midrule\n" + tab_latex.split("\\midrule")[1]
hacked_latex = before_header + custom_header + after_header
hacked_latex = hacked_latex.replace(
    "\\multirow[t]{3}{*}{3}", "\\midrule\n\\multirow[t]{3}{*}{3}"
)

In [39]:
with open("../docs/tables/experiment_speedups.tex", "w") as f:
    f.write(hacked_latex)

In [ ]:
tab2 = pd.DataFrame()
tab2[["$d$", "$p$"]] = speedup_table[["d", "p"]]
tab2["$\\mathcal{T}_h$"] = speedup_table["m"].apply(
    lambda m: f"${format_mesh((m, 'S'))}$"
)
tab2["DoFs"] = speedup_table["DoFs"].apply(lambda n: f"{n / 1_000_000:.1f}\\,{{M}}")

asm_parallel = "best ASM variant with $\\mathcal{T}_H = \\mathcal{T}_h$"
asm_classical = "best ASM variant with $\\mathcal{T}_H = \\mathcal{T}_\\mathcal{H}$"

tab2[(asm_classical, "$\\mathcal{T}_\\mathcal{H}$")] = speedup_table[
    "best_classical_meshing"
].map(lambda m: f"${format_mesh(m[0])}$")
tab2[(asm_classical, "$\\mathcal{T}_H$")] = speedup_table["best_classical_meshing"].map(
    lambda m: f"${format_mesh(m[1])}$"
)
tab2[(asm_classical, "time")] = speedup_table["best_classical_time"] / 1000
tab2[(asm_classical, "speedup")] = (
    speedup_table["best_amgx_time"] / tab2[(asm_classical, "time")] / 1000
)

tab2[(asm_parallel, "$\\mathcal{T}_\\mathcal{H}$")] = speedup_table[
    "best_parallel_meshing"
].map(lambda m: f"${format_mesh(m[0])}$")
tab2[(asm_parallel, "$\\mathcal{T}_H$")] = speedup_table["best_parallel_meshing"].map(
    lambda m: f"${format_mesh(m[1])}$"
)
tab2[(asm_parallel, "time")] = speedup_table["best_parallel_time"] / 1000
tab2[(asm_parallel, "speedup")] = (
    speedup_table["best_amgx_time"] / tab2[(asm_parallel, "time")] / 1000
)

tab2.set_index(["$d$", "$p$", "$\\mathcal{T}_h$", "DoFs"], inplace=True)

tab2[(asm_classical, "time")] = tab2[(asm_classical, "time")].apply(time_latex)
tab2[(asm_parallel, "time")] = tab2[(asm_parallel, "time")].apply(time_latex)
tab2[(asm_classical, "speedup")] = tab2[(asm_classical, "speedup")].apply(speedup_latex)
tab2[(asm_parallel, "speedup")] = tab2[(asm_parallel, "speedup")].apply(speedup_latex)

tab2.columns = pd.MultiIndex.from_tuples(tab2.columns)
tab2.rename(columns=lambda c: f"{{{c}}}", inplace=True)
tab2

TypeError: int() argument must be a string, a bytes-like object or a real number, not 'tuple'

In [ ]:
tab_latex = tab2.style.to_latex(
    hrules=True,
    multirow_align="t",
    multicol_align="c",
    column_format=f"rrc{dofs_col}|cc{time_col}{speedup_col}|cc{time_col}{speedup_col}",
)

In [ ]:
custom_header = r"""
 &  &  & & \multicolumn{4}{c|}{{best ASM variant with $\mathcal{T}_H = \mathcal{T}_\mathcal{H}$}} & \multicolumn{4}{c}{{best ASM variant with $\mathcal{T}_H = \mathcal{T}_h$}} \\
$d$ & $p$ & $\mathcal{T}_h$ & {DoFs} & {$\mathcal{T}_\mathcal{H}$} & {$\mathcal{T}_H$} & {time} & {speedup} & {$\mathcal{T}_\mathcal{H}$} & {$\mathcal{T}_H$} & {time} & {speedup} \\
"""
before_header = tab_latex.split("\\toprule")[0] + "\\toprule\n"
after_header = "\n\\midrule\n" + tab_latex.split("\\midrule")[1]
hacked_latex = before_header + custom_header + after_header
hacked_latex = hacked_latex.replace(
    "\\multirow[t]{3}{*}{3}", "\\midrule\n\\multirow[t]{3}{*}{3}"
)

In [ ]:
with open("../docs/tables/experiment_speedups_2.tex", "w") as f:
    f.write(hacked_latex)